# 3. Price a future on an index you built

Fair value from cost of carry, basis against a quote, implied repo, DV01 by
bump-and-revalue, and the cost of rolling.

Core library only — no optional extras.

## The idea in one line

A future is a claim on the index at expiry, so its fair value is the index
compounded at the financing rate and reduced by the dividends you forgo by not
holding the constituents:

$$F = S \times e^{(r - q)T}$$

Carry is $r - q$. When financing exceeds the dividend yield the future trades
**above** spot; when the yield exceeds financing it trades below. Nothing
mysterious happens at expiry — $T$ goes to zero and the future converges on
spot, which the last section shows directly.

## Setup

In [ ]:
import logging

import pandas as pd

from beacon.derivatives import IndexFuture
from beacon.derivatives.pricing import futures_roll_return, implied_repo_rate
from beacon.index.calculation import IndexCalculator
from beacon.index.constructor import IndexDefinition
from beacon.index.methodology import MarketCapWeighted
from beacon.synthetic import SyntheticConfig, generate

logging.basicConfig(level=logging.ERROR,
                    format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Step 1 — build the underlying

Any index will do. This is the same construction as notebook 01, collapsed to
one cell.

In [ ]:
CONFIG = SyntheticConfig(assets=40,
                         start="2021-01-04",
                         end="2024-12-31",
                         seed=3)

dataset = generate(CONFIG)

definition = IndexDefinition(
    index_id="FUTIDX",
    index_name="FUTIDX Index",
    base_date=CONFIG.start,
    base_value=1000.0,
    currency=CONFIG.currency,
    eligibility_rules=[],
    weighting_scheme=MarketCapWeighted(use_free_float=True),
    rebalancing_frequency="QUARTERLY",
    universe_identifiers=list(dataset.universe.index),
    max_constituent_weight=0.10)

index = IndexCalculator(definition, dataset.fetcher()).run(
    start_date=CONFIG.start, end_date=CONFIG.end)

spot = float(index.index_levels.iloc[-1])
valuation_date = pd.Timestamp(index.index_levels.index[-1])

print(f"{definition.index_id} at {spot:,.2f} on {valuation_date.date()}")

## Step 2 — define the contract

A contract is its expiry and its multiplier. The multiplier is what turns index
points into money: 50 currency units per point is the convention for a major
equity future, so a 4,000-point index is a 200,000 notional per contract.

In [ ]:
RISK_FREE = 0.045
DIVIDEND_YIELD = 0.018

# One index point is 50 currency units, the convention for a major equity
# future. Tick size and value follow from it.
MULTIPLIER = 50.0
TICK_SIZE = 0.25

expiry = valuation_date + pd.DateOffset(months=3)

future = IndexFuture(derivative_id="FUTIDX-Z",
                     underlying_id=definition.index_id,
                     currency=CONFIG.currency,
                     expiry_date=expiry.strftime("%Y-%m-%d"),
                     contract_multiplier=MULTIPLIER,
                     tick_size=TICK_SIZE,
                     tick_value=MULTIPLIER * TICK_SIZE)

pd.Series({"underlying": f"{future.underlying_id} at {spot:,.2f}",
           "valuation": str(valuation_date.date()),
           "expiry": str(expiry.date()),
           "time to expiry": f"{future.time_to_expiry(valuation_date):.4f} years (ACT/365)",
           "multiplier": f"{MULTIPLIER:,.0f} per index point",
           "notional": f"{spot * MULTIPLIER:,.0f} per contract"},
          name="contract")

## Step 3 — fair value

Financing costs 4.5%, the dividends you give up are worth 1.8%, so net carry is
positive and the future sits above spot.

In [ ]:
market = {"risk_free_rate": RISK_FREE, "dividend_yield": DIVIDEND_YIELD}

fair = future.fair_value(spot, valuation_date, market)
carry = RISK_FREE - DIVIDEND_YIELD

pd.Series({"risk free": f"{RISK_FREE:.2%}",
           "dividend yield": f"{DIVIDEND_YIELD:.2%}",
           "net carry": f"{carry:.2%}",
           "spot": f"{spot:,.2f}",
           "fair value": f"{fair:,.2f}",
           "premium": f"{fair - spot:+,.2f}  ({fair / spot - 1:+.3%})"},
          name="fair value")

### Carry can go either way

The premium is not a fact about futures, it is a fact about the two rates. Hold
spot fixed and vary them:

In [ ]:
scenarios = pd.DataFrame(
    [{"risk free": rate,
      "dividend yield": yield_,
      "net carry": rate - yield_,
      "fair value": future.fair_value(
          spot, valuation_date,
          {"risk_free_rate": rate, "dividend_yield": yield_}),
      }
     for rate, yield_ in [(0.045, 0.018), (0.045, 0.045),
                          (0.018, 0.045), (0.000, 0.030)]])

scenarios["premium to spot"] = scenarios["fair value"] - spot
scenarios.style.format({"risk free": "{:.2%}", "dividend yield": "{:.2%}",
                        "net carry": "{:+.2%}", "fair value": "{:,.2f}",
                        "premium to spot": "{:+,.2f}"})

Row two has financing exactly equal to the dividend yield, and the future prices
at spot: carrying the position costs precisely what holding the shares pays.
Row three inverts it and the future trades at a discount — which is the normal
state of affairs in a high-yield market or a low-rate one.

## Step 4 — basis against a quoted price

Fair value is what the model says. The basis is the difference between what the
market says and either spot or fair value.

In [ ]:
# A quote 15bp above fair value, so there is something to measure.
quoted = fair * 1.0015

implied = implied_repo_rate(futures_price=quoted,
                            spot=spot,
                            dividend_yield=DIVIDEND_YIELD,
                            time_to_expiry_years=future.time_to_expiry(valuation_date))

pd.Series({"quoted": f"{quoted:,.2f}",
           "fair": f"{fair:,.2f}",
           "basis to spot": f"{future.basis(quoted, spot):+,.2f}",
           "rich / cheap vs fair": f"{quoted - fair:+,.2f}",
           "annualised basis": f"{future.annualised_basis(quoted, spot, valuation_date):+.3%}",
           "implied repo": f"{implied:.3%}",
           "actual financing": f"{RISK_FREE:.3%}"},
          name="basis")

**Implied repo** is the financing rate the quoted price implies. It is the
basis expressed as a rate, which is what makes two contracts of different
maturities comparable — a 3-point basis on a one-month contract and a 12-point
basis on a one-year contract are the same trade.

Above your actual funding cost, buying the shares and selling the future
(cash-and-carry) earns the difference. Below it, the reverse.

## Step 5 — rate sensitivity

DV01 is computed by **bumping and revaluing**, not by differentiating a formula.

In [ ]:
ONE_BASIS_POINT = 0.0001

bumped = future.fair_value(
    spot, valuation_date,
    {**market, "risk_free_rate": RISK_FREE + ONE_BASIS_POINT})

dv01 = (bumped - fair) * MULTIPLIER

pd.Series({"fair value": f"{fair:,.4f}",
           "+1bp financing": f"{bumped:,.4f}",
           "difference": f"{bumped - fair:+,.4f} index points",
           "DV01": f"{dv01:+,.2f} per contract"},
          name="rate sensitivity")

That is one extra call to the same pricing function, so the sensitivity cannot
drift away from the pricer the way a hand-derived closed form does when someone
changes the day count.

## Step 6 — rolling to the next contract

A future expires. A position that must persist has to be rolled into the next
one, and positive carry makes the back contract dearer.

In [ ]:
far_expiry = expiry + pd.DateOffset(months=3)

far = IndexFuture(derivative_id="FUTIDX-H",
                  underlying_id=definition.index_id,
                  currency=CONFIG.currency,
                  expiry_date=far_expiry.strftime("%Y-%m-%d"),
                  contract_multiplier=MULTIPLIER,
                  tick_size=TICK_SIZE,
                  tick_value=MULTIPLIER * TICK_SIZE)

back_fair = far.fair_value(spot, valuation_date, market)
roll = future.roll_cost(fair, back_fair)

pd.Series({f"front ({expiry.date()})": f"{fair:,.2f}",
           f"back  ({far_expiry.date()})": f"{back_fair:,.2f}",
           "roll cost": f"{roll:+,.2f} index points",
           "per contract": f"{roll * MULTIPLIER:+,.2f}",
           "annualised roll": f"{futures_roll_return(fair, back_fair, expiry, far_expiry):+.3%}"},
          name="rolling")

That recurring cost is why a futures-based tracker drifts from the index it
follows even when it never mis-trades a single rebalance. Over four quarterly
rolls, the annualised figure above is what it gives up.

## Step 7 — convergence

The premium is entirely a function of remaining time. As expiry approaches, it
goes to zero and the future becomes the index.

In [ ]:
path = pd.DataFrame(
    [{"days to expiry": (expiry - date).days,
      "time to expiry": future.time_to_expiry(date),
      "fair value": future.fair_value(spot, date, market)}
     for date in pd.date_range(valuation_date, expiry, freq="14D")])

path["premium to spot"] = path["fair value"] - spot
path.set_index("days to expiry").style.format(
    {"time to expiry": "{:.4f}", "fair value": "{:,.2f}",
     "premium to spot": "{:+,.3f}"})

The last row is expiry itself: time to expiry is zero, and fair value is spot to
the last decimal. No convergence logic exists anywhere in the pricer — it falls
out of $e^{0} = 1$.

## Where to go next

- **`04_optimised_index.ipynb`** — optimise an index against real constraints